# Data Fetcher Module Testing Notebook

This notebook is for testing the `data_fetcher.py` module with your actual data.

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from imports import *
from ingestion_tables_multithreading import main as ingestion_tables_main
from ingestion_excels import main as ingestion_excels_main
from sharepoint import SharepointClient
from tqdm import tqdm
import contextlib
import io
import logging

# Add the parent directory's src folder to the path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir / 'src'))

# Import the data fetcher module
from data_fetcher import fetch_all_data

print(f"Added to path: {parent_dir / 'src'}")
print("Imports successful!")

Added to path: /Users/teq-admin/Downloads/otif_engine/src
Imports successful!


## 2. Load Your Data

Load credentials and data from either local files or SharePoint

In [2]:
def load_creds(path):
    creds = {}
    with open(path, 'r') as f:
        for line in f:
            if '=' in line:
                key, value = line.strip().split('=', 1)
                creds[key.strip()] = value.strip()
    return creds

creds = load_creds('creds.txt')
print("Credentials loaded successfully")

Credentials loaded successfully


In [3]:
# Choose data source method
method = "local"  # Change to "sharepoint" to fetch from SharePoint

In [4]:
if method == "local":
    print("Loading data from local files...")
    
    # Load tables
    csv_folder = "./local_data_dnd/tables/" 
    csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]
    dfs_tables = {}

    for file in csv_files:
        name = file.replace(".csv", "")
        file_path = os.path.join(csv_folder, file)

        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='latin1')

        dfs_tables[name] = df
    
    print(f"Loaded {len(dfs_tables)} tables")

    # Load excels
    csv_folder = "./local_data_dnd/excels" 
    csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]
    dfs_excels = {}
    
    for file in csv_files:
        name = file.replace(".csv", "")
        file_path = os.path.join(csv_folder, file)

        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='latin1')
            
        dfs_excels[name] = df
    
    print(f"Loaded {len(dfs_excels)} excel files")
    
else:
    print("Loading data from SharePoint...")
    dfs_tables = ingestion_tables_main(creds)
    dfs_excels = ingestion_excels_main(creds)

# Display loaded data
print(f"\nLoaded tables: {list(dfs_tables.keys())}")
print(f"\nLoaded excels: {list(dfs_excels.keys())}")
print(f"\nTotal tables: {len(dfs_tables)}")
print(f"Total excels: {len(dfs_excels)}")

Loading data from local files...
Loaded 12 tables
Loaded 27 excel files

Loaded tables: ['inb_data', 'po_data', 'dod_data', 'pi_ns_data', 'compliance_hubspot', 'pi_data', 'hs_codes_data', 'batch_data', 'pl_data', 'master_data', 'supplier_confirmation', 'telex_tableau']

Loaded excels: ['cprd', 'ffw_blockers', 'payrun', 'memo_mapping', 'buffer_mapping', 'blockers_mapping', 'fob_date', 'transparency_data', 'status_mapping', 'qc', 'asin_static_payment_status', 'ffw_status', 'prepayment', 'asin_priority_mapping', 'g2', 'booking_form_data', 'transparency_master', 'payment_terms_mapping', 'team_priority_mapping', 'spd_blockers', 'g4', 'compliance', 'telex_supplier', 'telex_ffw', 'cm_sm_vendor_mapping', 'packaging_data', 'prd']

Total tables: 12
Total excels: 27


## 3. Test Data Fetcher Module

In [5]:
# Run the data fetcher with debug mode on
try:
    final_df, mappings = fetch_all_data(dfs_tables, dfs_excels, debug=True)
    print("\n✅ Data fetching completed successfully!")
except Exception as e:
    print(f"\n❌ Error occurred: {str(e)}")
    import traceback
    traceback.print_exc()
    raise


[DEBUG] Starting data fetching process...
[DEBUG] Available tables: ['inb_data', 'po_data', 'dod_data', 'pi_ns_data', 'compliance_hubspot', 'pi_data', 'hs_codes_data', 'batch_data', 'pl_data', 'master_data', 'supplier_confirmation', 'telex_tableau']
[DEBUG] Available excels: ['cprd', 'ffw_blockers', 'payrun', 'memo_mapping', 'buffer_mapping', 'blockers_mapping', 'fob_date', 'transparency_data', 'status_mapping', 'qc', 'asin_static_payment_status', 'ffw_status', 'prepayment', 'asin_priority_mapping', 'g2', 'booking_form_data', 'transparency_master', 'payment_terms_mapping', 'team_priority_mapping', 'spd_blockers', 'g4', 'compliance', 'telex_supplier', 'telex_ffw', 'cm_sm_vendor_mapping', 'packaging_data', 'prd']

[DEBUG] Base PO data shape: (2285, 44)
[DEBUG] PO data columns: ['id', 'date_created', 'document_number', 'subsidiary_no_hierarchy', 'scm_associated_brands', 'po_vendor', 'supplier_confirmation_status', 'final_status', 'scm_po_scm_memo', 'marketplace_header']...

[DEBUG] Start

## 4. Explore the Results

In [6]:
# Basic information about the result
print(f"Final dataframe shape: {final_df.shape}")
print(f"\nColumns in final dataframe: {final_df.shape[1]}")
print(f"Rows in final dataframe: {final_df.shape[0]}")
print(f"\nMappings returned: {list(mappings.keys())}")

Final dataframe shape: (2285, 105)

Columns in final dataframe: 105
Rows in final dataframe: 2285

Mappings returned: ['status_mapping', 'blockers_mapping', 'payment_terms_mapping', 'asin_priority_mapping', 'team_priority_mapping']


In [7]:
# Display first few rows
print("First 5 rows of the consolidated dataframe:")
final_df.head()

First 5 rows of the consolidated dataframe:


,id,date_created,document_number,subsidiary_no_hierarchy,scm_associated_brands,po_vendor,supplier_confirmation_status,final_status,scm_po_scm_memo,marketplace_header,...,Joey Status,Telex Supplier Action,Muazam Status,Telex FFW Blocker,Packaging Status,Packaging Standard Status,FOB Status,FFW Blocker Status,status_mapping_available,blockers_mapping_available
0,25706864,2023-06-21,PO356828,Razor Group Procurement Ltd,Enno Vatti,"72375 Verslo Linija, UAB",Confirmed,Partially Received,Q4_Batch_5_Automated_PO,UK,...,Not Released,Not in Telex Sheet,Not Released,Not in FFW Telex Sheet,Yes,06b. SCM Check Pending,,No,True,True
1,32304245,2024-01-08,PO359395,RGA Hawwwy LLC,MeantToBe,"72328 Market Union Co.,Ltd",Confirmed,Pending Billing/Partially Received,2025 - Moved from 2024 --- US Consolidation Tr...,US,...,Not Released,Not in Telex Sheet,Not Released,Not in FFW Telex Sheet,No,,,No,True,True
2,32304245,2024-01-08,PO359395,RGA Hawwwy LLC,MeantToBe,"72328 Market Union Co.,Ltd",Confirmed,Pending Billing/Partially Received,2025 - Moved from 2024 --- US Consolidation Tr...,US,...,Not Released,Not in Telex Sheet,Not Released,Not in FFW Telex Sheet,No,,,No,True,True
3,32304245,2024-01-08,PO359395,RGA Hawwwy LLC,MeantToBe,"72328 Market Union Co.,Ltd",Confirmed,Pending Billing/Partially Received,2025 - Moved from 2024 --- US Consolidation Tr...,US,...,Not Released,Not in Telex Sheet,Not Released,Not in FFW Telex Sheet,No,,,No,True,True
4,32916541,2024-01-23,PO360234,RGA FFM LLC,Dawhud Direct,"70673 FUJIAN QUANZHOU PENGHONG ARTWARE CO.,LTD",Confirmed,Partially Received,Q2_Batch_2_Automated_PO_2024,CA,...,Not Released,Not in Telex Sheet,Not Released,Not in FFW Telex Sheet,Yes,06b. SCM Check Pending,,No,True,True


In [8]:
# Check key columns that should have been created
key_columns = ['po_razin', 'po_razin_id', 'razin_mp', 'asin_mp', 'Vendor ID']
print("Checking key columns:")
for col in key_columns:
    if col in final_df.columns:
        print(f"✅ {col} - Found")
    else:
        print(f"❌ {col} - Missing")

Checking key columns:
✅ po_razin - Found
✅ po_razin_id - Found
✅ razin_mp - Found
✅ asin_mp - Found
✅ Vendor ID - Found


In [9]:
# Check mapped fields
mapped_fields = [
    'Placement Batch', 'Supplier Confirmation VP Check', 'NS PI Status', 'VP PI Status',
    'PI Payment Status', 'INB#', 'HS Code', 'Batch Sign-Off', 'CM', 'SM', 'Team',
    'Compliance Status', 'Transparency Check', 'OTIF Focus', 'L2 SPD', 'L2 Compliance'
]

print("Checking mapped fields:")
for field in mapped_fields:
    if field in final_df.columns:
        non_empty = (final_df[field] != "").sum()
        print(f"✅ {field:<30} - Found ({non_empty} non-empty values)")
    else:
        print(f"❌ {field:<30} - Missing")

Checking mapped fields:
✅ Placement Batch                - Found (2285 non-empty values)
✅ Supplier Confirmation VP Check - Found (2285 non-empty values)
✅ NS PI Status                   - Found (2285 non-empty values)
✅ VP PI Status                   - Found (2285 non-empty values)
✅ PI Payment Status              - Found (2285 non-empty values)
✅ INB#                           - Found (1043 non-empty values)
✅ HS Code                        - Found (2285 non-empty values)
✅ Batch Sign-Off                 - Found (2285 non-empty values)
✅ CM                             - Found (2285 non-empty values)
✅ SM                             - Found (2285 non-empty values)
✅ Team                           - Found (2285 non-empty values)
✅ Compliance Status              - Found (2285 non-empty values)
✅ Transparency Check             - Found (2285 non-empty values)
✅ OTIF Focus                     - Found (2285 non-empty values)
✅ L2 SPD                         - Found (2285 non-empty values)
✅

In [10]:
# Display all column names grouped by category
print("All columns in the final dataframe:\n")

# Original PO columns
po_columns = [col for col in dfs_tables.get('po_data', pd.DataFrame()).columns if col in final_df.columns]
print(f"Original PO columns ({len(po_columns)}):")
print(po_columns[:10], "..." if len(po_columns) > 10 else "")

# New mapped columns
new_columns = [col for col in final_df.columns if col not in po_columns]
print(f"\nNew mapped columns ({len(new_columns)}):")
for i in range(0, len(new_columns), 5):
    print(new_columns[i:i+5])

All columns in the final dataframe:

Original PO columns (44):
['id', 'date_created', 'document_number', 'subsidiary_no_hierarchy', 'scm_associated_brands', 'po_vendor', 'supplier_confirmation_status', 'final_status', 'scm_po_scm_memo', 'marketplace_header'] ...

New mapped columns (61):
['po_razin', 'razin_mp', 'asin_mp', 'Vendor ID', 'Placement Batch']
['Supplier Confirmation VP Check', 'NS PI Status', 'VP PI Status', 'PI Payment Status', 'INB#']
['Status', 'Actual Pickup', 'Actual Shipping Date3', 'Actual Arrival Date', 'Actual Delivery Date']
['Expected Arrival Date', 'Substatus', 'Shipment Method', 'HS Code', 'Actual pick-up date']
['Gate In Date', 'Actual Shipping Date', 'FOB Date', 'Incoterms2', 'SPD']
['SPD Delay Reason', 'Batch Sign-Off', 'CM', 'SM', 'Team']
['razin_mp_vendor', 'Compliance Status', 'Transparency Check', 'Transparency Pending', 'OTIF Focus']
['MD Blocker', 'L2 SPD', 'L2 Compliance', 'L2 PI', 'L2 PRD']
['L2 CPRD', 'L2 G2', 'L2 G4', 'L2 Pickup', 'L2 QC']
['Line P

## 5. Data Quality Checks

In [11]:
# Check for missing document numbers
missing_doc_nums = (final_df['document_number'] == "").sum()
print(f"Rows with missing document numbers: {missing_doc_nums}")

# Check for missing batch IDs
missing_batch_ids = (final_df['batch_id'] == "").sum()
print(f"Rows with missing batch IDs: {missing_batch_ids}")

# Check for missing INB#
missing_inb = (final_df['INB#'] == "").sum()
print(f"Rows with missing INB#: {missing_inb}")

Rows with missing document numbers: 0
Rows with missing batch IDs: 919
Rows with missing INB#: 1242


In [12]:
# Check data types of key columns
print("Data types of key columns:")
key_cols = ['document_number', 'Vendor ID', 'quantity', 'item_rate_eur', 'batch_id']
for col in key_cols:
    if col in final_df.columns:
        print(f"{col}: {final_df[col].dtype}")

Data types of key columns:
document_number: object
Vendor ID: Int64
quantity: float64
item_rate_eur: float64
batch_id: object


In [13]:
# Sample specific mapped data
print("Sample of key mapped fields:")
sample_cols = ['document_number', 'item', 'Vendor ID', 'CM', 'SM', 'Compliance Status', 'INB#']
available_cols = [col for col in sample_cols if col in final_df.columns]
if available_cols:
    display(final_df[available_cols].head(10))

Sample of key mapped fields:


,document_number,item,Vendor ID,CM,SM,Compliance Status,INB#
0,PO356828,ENNO-000005,72375,Darren Fernandes,Darren Fernandes,Approved,
1,PO359395,M2B4-000449,72328,Hilfee Lu,Vivian Gao,Approved,
2,PO359395,M2B4-100075,72328,Hilfee Lu,Vivian Gao,To Be Tested,
3,PO359395,M2B4-100253,72328,Hilfee Lu,Vivian Gao,Approved,
4,PO360234,DAWH-000891,70673,Paul Fong,Tinia Zhang,Approved,
5,PO362894,REGA-000005,70302,Angela Chen,Vivian Gao,Approved,
6,PO363283,EVER-000175,71529,Jeremy Lin,Lemon Shen,To Be Tested,
7,PO363527,NOES-000002,70471,Angela Chen,Teresa Xiong,Approved,
8,PO363613,AMZY-000055,73911,Angela Chen,Maggie Yang,Approved,
9,PO363613,AMZY-000060,73911,Angela Chen,Maggie Yang,Approved,


## 6. Export Results (Optional)

In [19]:
# Uncomment to save the consolidated dataframe
final_df.to_csv('consolidated_data.csv', index=False)
print("Data saved to consolidated_data.csv")

Data saved to consolidated_data.csv


## 7. Test Specific Mappings

In [15]:
# Test a specific document number
test_doc_num = final_df['document_number'].iloc[0] if len(final_df) > 0 else None
if test_doc_num and test_doc_num != "":
    print(f"Testing document number: {test_doc_num}")
    test_row = final_df[final_df['document_number'] == test_doc_num].iloc[0]
    
    print("\nMapped values for this document:")
    print(f"NS PI Status: {test_row.get('NS PI Status', 'N/A')}")
    print(f"VP PI Status: {test_row.get('VP PI Status', 'N/A')}")
    print(f"Supplier Confirmation VP Check: {test_row.get('Supplier Confirmation VP Check', 'N/A')}")
    print(f"Placement Batch: {test_row.get('Placement Batch', 'N/A')}")

Testing document number: PO356828

Mapped values for this document:
NS PI Status: Paid In Full
VP PI Status: 04a. SM Review Pending
Supplier Confirmation VP Check: Available on VP
Placement Batch: Other


In [16]:
# Check unique values for categorical fields
categorical_fields = ['Placement Batch', 'Compliance Status', 'HS Code', 'Batch Sign-Off']
for field in categorical_fields:
    if field in final_df.columns:
        unique_vals = final_df[field].value_counts().head(10)
        print(f"\n{field} - Top 10 values:")
        print(unique_vals)


Placement Batch - Top 10 values:
Placement Batch
Other                1347
2025 Q4 - Batch 3     420
2025 Q4 - Batch 1     287
2025 Q4 - Batch 2     122
2025 Q3 - Batch 2      47
2025 Q3 - Batch 1      43
2025 Q1 - Batch 4       6
2025 Q1 - Batch 3       5
2025 Q1 - Batch 2       4
2025 Q1 - Batch 1       2
Name: count, dtype: int64

Compliance Status - Top 10 values:
Compliance Status
Approved        2222
To Be Tested      40
Blocked           15
Missing            8
Name: count, dtype: int64

HS Code - Top 10 values:
HS Code
Available          2097
HS Code Missing     188
Name: count, dtype: int64

Batch Sign-Off - Top 10 values:
Batch Sign-Off
Signed-Off                1334
14a. Documents Missing     951
Name: count, dtype: int64


## 8. Performance Check

In [17]:
# Time the data fetcher without debug mode
import time

start_time = time.time()
final_df_perf, _ = fetch_all_data(dfs_tables, dfs_excels, debug=False)
end_time = time.time()

print(f"Data fetching took {end_time - start_time:.2f} seconds without debug mode")
print(f"Processing {len(final_df_perf):,} rows")

Data fetching took 0.38 seconds without debug mode
Processing 2,285 rows


## 9. Verify Required Tables

In [18]:
# Check if all required tables are loaded
required_tables = [
    'po_data', 'pl_data', 'batch_data', 'inb_data', 'telex_tableau',
    'pi_data', 'pi_ns_data', 'supplier_confirmation', 'master_data',
    'compliance_hubspot', 'hs_codes_data'
]

required_excels = [
    'memo_mapping', 'status_mapping', 'blockers_mapping', 'cm_sm_vendor_mapping',
    'asin_priority_mapping', 'payment_terms_mapping', 'team_priority_mapping',
    'asin_static_payment_status', 'ffw_status', 'fob_date', 'spd_blockers',
    'ffw_blockers', 'telex_supplier', 'telex_ffw', 'payrun', 'packaging_data',
    'transparency_data', 'transparency_master', 'prepayment', 'prd', 'cprd',
    'g2', 'g4', 'qc', 'compliance', 'booking_form_data'
]

print("Checking required tables:")
for table in required_tables:
    if table in dfs_tables:
        shape = dfs_tables[table].shape
        print(f"✅ {table:<25} - Loaded ({shape[0]} rows, {shape[1]} cols)")
    else:
        print(f"❌ {table:<25} - Missing")

print("\nChecking required excel files:")
for excel in required_excels:
    if excel in dfs_excels:
        shape = dfs_excels[excel].shape
        print(f"✅ {excel:<25} - Loaded ({shape[0]} rows, {shape[1]} cols)")
    else:
        print(f"❌ {excel:<25} - Missing")

Checking required tables:
✅ po_data                   - Loaded (2285 rows, 44 cols)
✅ pl_data                   - Loaded (7880 rows, 2 cols)
✅ batch_data                - Loaded (1684 rows, 16 cols)
✅ inb_data                  - Loaded (1633 rows, 29 cols)
✅ telex_tableau             - Loaded (14271 rows, 7 cols)
✅ pi_data                   - Loaded (11811 rows, 5 cols)
✅ pi_ns_data                - Loaded (2271 rows, 4 cols)
✅ supplier_confirmation     - Loaded (12515 rows, 4 cols)
✅ master_data               - Loaded (2614 rows, 7 cols)
✅ compliance_hubspot        - Loaded (10667 rows, 9 cols)
✅ hs_codes_data             - Loaded (20261 rows, 6 cols)

Checking required excel files:
✅ memo_mapping              - Loaded (12 rows, 2 cols)
✅ status_mapping            - Loaded (88 rows, 6 cols)
✅ blockers_mapping          - Loaded (206 rows, 3 cols)
✅ cm_sm_vendor_mapping      - Loaded (1031 rows, 6 cols)
✅ asin_priority_mapping     - Loaded (70 rows, 3 cols)
✅ payment_terms_mapping     -